In [ ]:

INPUT_LABELS_PATH = "/kaggle/input/datasets/harshiikkaa/foodev-v3-13760/test_v3_13760.csv"   
OUTPUT_DIR        = "/kaggle/working"
BATCH_SIZE        = 16      
MAX_SEQ_LEN       = 1024    


SEQ_COL      = "sequence"
ID_COL       = "seq_id"                              
LABEL_COLS   = ["primary_label", "secondary_label"]  


import os, gc, sys, json, time, warnings
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:


def load_dataset(path):
    df = pd.read_csv(path)
    print(f"Loaded {len(df)} rows, columns: {list(df.columns)}")

    missing = [c for c in [SEQ_COL, ID_COL] + LABEL_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected column(s): {missing}. "
                         f"Found: {list(df.columns)}")

    if df[ID_COL].duplicated().any():
        raise ValueError(f"'{ID_COL}' must be unique - found duplicates.")

    sequences = df[SEQ_COL].astype(str).str.strip().str.upper().tolist()
    meta      = df[[ID_COL] + LABEL_COLS].reset_index(drop=True)

    print(f"Sequence col: '{SEQ_COL}'  |  ID col: '{ID_COL}'  |  "
          f"Label cols: {LABEL_COLS}")
    print(f"Sequence length stats - min: {min(len(s) for s in sequences)}, "
          f"max: {max(len(s) for s in sequences)}, "
          f"mean: {np.mean([len(s) for s in sequences]):.0f}")
    for c in LABEL_COLS:
        print(f"  {c}: {df[c].value_counts().sort_index().to_dict()}")
    return sequences, meta, df

sequences, meta, df_orig = load_dataset(INPUT_LABELS_PATH)
N = len(sequences)

n_trunc = sum(len(s) > MAX_SEQ_LEN for s in sequences)
if n_trunc:
    print(f"  !! {n_trunc} sequences exceed MAX_SEQ_LEN={MAX_SEQ_LEN} and will be truncated")


split_idx_path = os.path.join(OUTPUT_DIR, "split_index.csv")
if not os.path.exists(split_idx_path):
    meta.assign(global_index=range(N)).to_csv(split_idx_path, index=False)
    print(f"Saved: {split_idx_path}")


In [ ]:


def done_path(plm_name):
    return os.path.join(OUTPUT_DIR, f".{plm_name}.done")

def out_path(plm_name):
    return os.path.join(OUTPUT_DIR, f"features_{plm_name}_{N}.csv")

def is_done(plm_name):
    return os.path.exists(done_path(plm_name)) and os.path.exists(out_path(plm_name))

def mark_done(plm_name):
    with open(done_path(plm_name), "w") as f:
        f.write(time.strftime("%Y-%m-%d %H:%M:%S"))

def save_features(plm_name, embeddings: np.ndarray):
    """Save embeddings with seq_id and both label columns in front. The
    ID/label columns are attached after the forward pass, by row position,
    which is safe because embed_in_batches preserves input order exactly.
    """
    assert embeddings.shape[0] == N, \
        f"Row count mismatch: got {embeddings.shape[0]}, expected {N}"
    cols = [f"dim_{i}" for i in range(embeddings.shape[1])]
    emb_df = pd.DataFrame(embeddings, columns=cols)
    out_df = pd.concat([meta.reset_index(drop=True), emb_df], axis=1)
    out_df.to_csv(out_path(plm_name), index=False)
    mark_done(plm_name)
    print(f"  Saved {out_path(plm_name)}  shape={out_df.shape} "
          f"({embeddings.shape[1]} dims + {meta.shape[1]} id/label cols)")

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


In [ ]:


def embed_in_batches(encode_fn, sequences, batch_size=BATCH_SIZE, desc=""):
    """
    encode_fn(batch: List[str]) -> np.ndarray of shape (len(batch), D)
    Returns full (N, D) array.
    """
    all_embs = []
    n = len(sequences)
    for start in range(0, n, batch_size):
        batch = sequences[start : start + batch_size]
        emb   = encode_fn(batch)
        all_embs.append(emb)
        done  = min(start + batch_size, n)
        print(f"\r  {desc} {done}/{n}", end="", flush=True)
    print()
    return np.vstack(all_embs)




In [ ]:


PLM_NAME = "esm2_150M"

if is_done(PLM_NAME):
    print(f"[SKIP] {PLM_NAME} already done.")
else:
    print(f"\n{'='*60}\n  ESM2 (esm2_t30_150M_UR50D)\n{'='*60}")
    from transformers import AutoTokenizer, EsmModel

    tok = AutoTokenizer.from_pretrained("facebook/esm2_t30_150M_UR50D")
    mdl = EsmModel.from_pretrained("facebook/esm2_t30_150M_UR50D").eval().to(DEVICE)

    def encode_esm2(batch):
        # Truncate to MAX_SEQ_LEN (tokenizer does this, but be explicit)
        seqs = [s[:MAX_SEQ_LEN] for s in batch]
        enc  = tok(seqs, return_tensors="pt", padding=True,
                   truncation=True, max_length=MAX_SEQ_LEN).to(DEVICE)
        with torch.no_grad():
            out = mdl(**enc)
        # Mean-pool over the sequence dimension 
        mask = enc["attention_mask"].unsqueeze(-1).float()
        emb  = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
        return emb.cpu().float().numpy()

    embs = embed_in_batches(encode_esm2, sequences, desc="ESM2_150M")
    save_features(PLM_NAME, embs)

    del tok, mdl, embs
    free_gpu()

In [ ]:


PLM_NAME = "protbert"

if is_done(PLM_NAME):
    print(f"[SKIP] {PLM_NAME} already done.")
else:
    print(f"\n{'='*60}\n  ProtBERT (Rostlab/prot_bert)\n{'='*60}")
    from transformers import BertTokenizer, BertModel

    tok = BertTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
    mdl = BertModel.from_pretrained("Rostlab/prot_bert").eval().to(DEVICE)

    def encode_protbert(batch):
        # ProtBERT expects space-separated amino acids
        seqs = [" ".join(list(s[:MAX_SEQ_LEN])) for s in batch]
        enc  = tok(seqs, return_tensors="pt", padding=True,
                   truncation=True, max_length=MAX_SEQ_LEN + 2).to(DEVICE)
        with torch.no_grad():
            out = mdl(**enc)
        mask = enc["attention_mask"].unsqueeze(-1).float()
        emb  = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
        return emb.cpu().float().numpy()

    embs = embed_in_batches(encode_protbert, sequences, batch_size=8, desc="ProtBERT")
    save_features(PLM_NAME, embs)

    del tok, mdl, embs
    free_gpu()


In [ ]:


PLM_NAME = "protgpt2"

if is_done(PLM_NAME):
    print(f"[SKIP] {PLM_NAME} already done.")
else:
    print(f"\n{'='*60}\n  ProtGPT2 (nferruz/ProtGPT2)\n{'='*60}")
    from transformers import AutoTokenizer, GPT2Model

    tok = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
    
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    mdl = GPT2Model.from_pretrained("nferruz/ProtGPT2").eval().to(DEVICE)
    mdl.config.pad_token_id = tok.eos_token_id

    def encode_protgpt2(batch):
        seqs = [s[:MAX_SEQ_LEN] for s in batch]
        enc  = tok(seqs, return_tensors="pt", padding=True,
                   truncation=True, max_length=MAX_SEQ_LEN).to(DEVICE)
        with torch.no_grad():
            out = mdl(**enc)
        
        hidden = out.last_hidden_state                  # (B, L, D)
        mask   = enc["attention_mask"].unsqueeze(-1).float()
        emb    = (hidden * mask).sum(1) / mask.sum(1)
        return emb.cpu().float().numpy()

    embs = embed_in_batches(encode_protgpt2, sequences, batch_size=8, desc="ProtGPT2")
    save_features(PLM_NAME, embs)

    del tok, mdl, embs
    free_gpu()


In [ ]:


PLM_NAME = "prott5"

if is_done(PLM_NAME):
    print(f"[SKIP] {PLM_NAME} already done.")
else:
    print(f"\n{'='*60}\n  ProtT5-XL (Rostlab/prot_t5_xl_uniref50)\n{'='*60}")
    from transformers import T5Tokenizer, T5EncoderModel

    tok = T5Tokenizer.from_pretrained("Rostlab/prot_t5_xl_uniref50", do_lower_case=False)
    mdl = T5EncoderModel.from_pretrained(
        "Rostlab/prot_t5_xl_uniref50", torch_dtype=torch.float16   # fp16 to fit T4
    ).eval().to(DEVICE)

    def encode_prott5(batch):
        
        seqs = [" ".join(list(s[:MAX_SEQ_LEN].upper().replace("U","X").replace("Z","X").replace("O","X").replace("B","X")))
                for s in batch]
        enc  = tok(seqs, return_tensors="pt", padding=True,
                   truncation=True, max_length=MAX_SEQ_LEN + 1).to(DEVICE)
        with torch.no_grad():
            out = mdl(**enc)
        hidden = out.last_hidden_state.float()          # back to float32
        mask   = enc["attention_mask"].unsqueeze(-1).float()
        emb    = (hidden * mask).sum(1) / mask.sum(1)
        return emb.cpu().numpy()

    embs = embed_in_batches(encode_prott5, sequences, batch_size=4, desc="ProtT5")
    save_features(PLM_NAME, embs)

    del tok, mdl, embs
    free_gpu()


In [ ]:

PLM_NAME = "ankh"

if is_done(PLM_NAME):
    print(f"[SKIP] {PLM_NAME} already done.")
else:
    print(f"\n{'='*60}\n  Ankh-base (ElnaggarLab/ankh-base)\n{'='*60}")

    import sys, importlib, subprocess

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "ankh", "-q"],
        check=True,
    )
    importlib.invalidate_caches()

    # sentencepiece >= 0.2.0 requires Load() to receive a plain str, but
    # transformers passes a pathlib.Path. Patch it to coerce the path.
    import sentencepiece as _spm

    _orig_spm_load = _spm.SentencePieceProcessor.Load

    def _patched_spm_load(self, model_file=None, model_proto=None):
        if model_file is not None:
            model_file = str(model_file)
        return _orig_spm_load(self, model_file=model_file, model_proto=model_proto)

    _spm.SentencePieceProcessor.Load = _patched_spm_load
    print("  [patch] sentencepiece.Load() -> coerces Path to str")

    # Load model (ankh-lib preferred, transformers fallback)
    use_ankh_lib = False

    try:
        import ankh as ankh_lib
        tok, mdl = ankh_lib.load_base_model()
        mdl = mdl.eval().to(DEVICE)
        use_ankh_lib = True
        print("  Loaded via ankh package")

    except Exception as e:
        print(f"  ankh package failed ({e}), falling back to T5EncoderModel")
        from transformers import AutoTokenizer, T5EncoderModel

        tok = AutoTokenizer.from_pretrained("ElnaggarLab/ankh-base")
        mdl = (
            T5EncoderModel                              # encoder-only, no decoder needed
            .from_pretrained("ElnaggarLab/ankh-base")
            .eval()
            .to(DEVICE)
        )
        print("  Loaded via transformers (T5EncoderModel)")

    # Encoding function
    def encode_ankh(batch: list[str]) -> "np.ndarray":
        """Mean-pool Ankh-base hidden states over non-padding positions.
        ankh-lib uses a character-level tokenizer (is_split_into_words=True);
        transformers' AutoTokenizer handles plain strings directly."""
        seqs = [s[:MAX_SEQ_LEN] for s in batch]        # hard-cap before tokenizer

        if use_ankh_lib:
            enc = tok(
                [list(s) for s in seqs],                # char-level split required
                is_split_into_words=True,
                add_special_tokens=True,
                padding=True,
                truncation=True,
                max_length=MAX_SEQ_LEN + 2,             # +2 for <BOS>/<EOS>
                return_tensors="pt",
            ).to(DEVICE)
        else:
            enc = tok(
                seqs,
                add_special_tokens=True,
                padding=True,
                truncation=True,
                max_length=MAX_SEQ_LEN + 2,
                return_tensors="pt",
            ).to(DEVICE)

        with torch.no_grad():
            out = mdl(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
            )

        hidden = out.last_hidden_state                          # (B, L, H)
        mask   = enc["attention_mask"].unsqueeze(-1).float()    # (B, L, 1)
        emb    = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

        return emb.cpu().float().numpy()

    embs = embed_in_batches(
        encode_ankh,
        sequences,
        batch_size=8,
        desc="Ankh-base",
    )
    save_features(PLM_NAME, embs)

    del tok, mdl, embs
    free_gpu()

In [ ]:

PLM_NAME = "esm2_650M"

if is_done(PLM_NAME):
    print(f"[SKIP] {PLM_NAME} already done.")
else:
    print(f"\n{'='*60}\n  ESM2-650M (esm2_t33_650M_UR50D)\n{'='*60}")
    from transformers import AutoTokenizer, EsmModel

    tok = AutoTokenizer.from_pretrained("facebook/esm2_t33_650M_UR50D")
    mdl = EsmModel.from_pretrained("facebook/esm2_t33_650M_UR50D").eval().to(DEVICE)

    def encode_esm2_650(batch):
        seqs = [s[:MAX_SEQ_LEN] for s in batch]
        enc  = tok(seqs, return_tensors="pt", padding=True,
                   truncation=True, max_length=MAX_SEQ_LEN).to(DEVICE)
        with torch.no_grad():
            out = mdl(**enc)
        mask = enc["attention_mask"].unsqueeze(-1).float()
        emb  = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
        return emb.cpu().float().numpy()

    embs = embed_in_batches(encode_esm2_650, sequences,
                            batch_size=8, desc="ESM2-650M")
    save_features(PLM_NAME, embs)

    del tok, mdl, embs
    free_gpu()

In [ ]:


print("\n" + "="*60)
print("  ALL PLMs COMPLETE")
print("="*60)

PLMS = ["esm2", "protbert", "protgpt2", "prott5", "ankh", "esm2_650M", "prot_electra"]

completed = []
for name in PLMS:
    status = "OK" if is_done(name) else "MISSING"
    path   = out_path(name)
    size   = f"{os.path.getsize(path)/1e6:.1f} MB" if os.path.exists(path) else "-"
    print(f"  {status:<8}  {name:<14}  {size}")
    if is_done(name):
        completed.append(name)

print(f"\nCompleted: {len(completed)}/{len(PLMS)}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Rows per file: {N}   Columns: seq_id, primary_label, secondary_label, dim_0...")


In [ ]:
!pip install biopython -q

In [ ]:


import os, re, math, warnings
from collections import Counter
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

from Bio.SeqUtils.ProtParam import ProteinAnalysis

# ── CONFIG ────────────────────────────────────────────────────────────────────
INPUT_PATH  = "/kaggle/input/datasets/harshiikkaa/foodev-7059/train_7059.csv"   
OUTPUT_DIR  = "/kaggle/working"
SEQ_COL     = "sequence"
ID_COL      = "seq_id"
META_COLS   = ["seq_id", "primary_label", "secondary_label"]

INCLUDE_DPC = True     # 400 dipeptide columns
os.makedirs(OUTPUT_DIR, exist_ok=True)

AA = "ACDEFGHIKLMNPQRSTVWY"

# ── SCALES ────────────────────────────────────────────────────────────────────
KD = {"A":1.8,"R":-4.5,"N":-3.5,"D":-3.5,"C":2.5,"Q":-3.5,"E":-3.5,"G":-0.4,
      "H":-3.2,"I":4.5,"L":3.8,"K":-3.9,"M":1.9,"F":2.8,"P":-1.6,"S":-0.8,
      "T":-0.7,"W":-0.9,"Y":-1.3,"V":4.2}                        # Kyte-Doolittle

# Chou-Fasman conformational parameters
P_HELIX = {"A":1.42,"C":0.70,"D":1.01,"E":1.51,"F":1.13,"G":0.57,"H":1.00,
           "I":1.08,"K":1.16,"L":1.21,"M":1.45,"N":0.67,"P":0.57,"Q":1.11,
           "R":0.98,"S":0.77,"T":0.83,"V":1.06,"W":1.08,"Y":0.69}
P_SHEET = {"A":0.83,"C":1.19,"D":0.54,"E":0.37,"F":1.38,"G":0.75,"H":0.87,
           "I":1.60,"K":0.74,"L":1.30,"M":1.05,"N":0.89,"P":0.55,"Q":1.10,
           "R":0.93,"S":0.75,"T":1.19,"V":1.70,"W":1.37,"Y":1.47}
P_TURN  = {"A":0.66,"C":1.19,"D":1.46,"E":0.74,"F":0.60,"G":1.56,"H":0.95,
           "I":0.47,"K":1.01,"L":0.59,"M":0.60,"N":1.56,"P":1.52,"Q":0.98,
           "R":0.95,"S":1.43,"T":0.96,"V":0.50,"W":0.96,"Y":1.14}

# Residue classes
HYDROPHOBIC = set("AVILMFWC")
POLAR       = set("STNQY")
CHARGED     = set("DEKRH")
POSITIVE    = set("KRH")
NEGATIVE    = set("DE")
AROMATIC    = set("FWY")
TINY        = set("AGCS")
SMALL       = set("AGCSTPDNV")

DISORDER    = set("SPEKQAG")


RED_SCHEMES = {
    "hp"  : {"H": "AVILMFWCY", "P": "GSTNQDEKRHP"},                  # 2-letter
    "chem": {"aliphatic": "AVILM", "aromatic": "FWY", "polar": "STNQ",
             "positive": "KRH", "negative": "DE", "special": "GPC"},  # 6-letter
}

MW_WATER = 18.015
MW_AA = {"A":89.09,"R":174.20,"N":132.12,"D":133.10,"C":121.16,"E":147.13,
         "Q":146.15,"G":75.07,"H":155.16,"I":131.17,"L":131.17,"K":146.19,
         "M":149.21,"F":165.19,"P":115.13,"S":105.09,"T":119.12,"W":204.23,
         "Y":181.19,"V":117.15}


PKA = {"C": 8.5, "D": 3.9, "E": 4.1, "H": 6.0, "K": 10.5, "R": 12.5, "Y": 10.1,
       "Nterm": 8.0, "Cterm": 3.1}


# ── HELPERS ───────────────────────────────────────────────────────────────────
def net_charge(seq, pH):
    """Henderson-Hasselbalch net charge at a given pH.

    Evaluated across the GI transit range because that is the paper's premise:
    stomach ~2, small intestine ~7, milk ~6.7. A protein's charge behaviour
    across that range is not something the negatives were matched on.
    """
    c = Counter(seq)
    pos = 1.0 / (1.0 + 10 ** (pH - PKA["Nterm"]))
    for aa in "KRH":
        pos += c[aa] / (1.0 + 10 ** (pH - PKA[aa]))
    neg = 1.0 / (1.0 + 10 ** (PKA["Cterm"] - pH))
    for aa in "DECY":
        neg += c[aa] / (1.0 + 10 ** (PKA[aa] - pH))
    return pos - neg


def shannon_entropy(seq):
    n = len(seq)
    if n == 0:
        return 0.0
    c = Counter(seq)
    return -sum((v/n) * math.log2(v/n) for v in c.values())


def longest_run(seq):
    best = run = 1
    for i in range(1, len(seq)):
        run = run + 1 if seq[i] == seq[i-1] else 1
        best = max(best, run)
    return best


def low_complexity_frac(seq, w=20, thr=3.0):
    """Fraction of 20-residue windows whose Shannon entropy falls below 3 bits.
    A cheap stand-in for SEG-style low-complexity detection."""
    if len(seq) < w:
        return float(shannon_entropy(seq) < thr)
    lows = sum(shannon_entropy(seq[i:i+w]) < thr for i in range(0, len(seq)-w+1, 5))
    tot  = len(range(0, len(seq)-w+1, 5))
    return lows / tot if tot else 0.0


def max_hydrophobic_window(seq, w=19):
    """Highest mean Kyte-Doolittle score over a 19-residue window — the standard
    transmembrane-helix proxy. 19 residues is the span needed to cross a lipid
    bilayer."""
    if len(seq) < w:
        return float(np.mean([KD.get(a, 0) for a in seq])) if seq else 0.0
    vals = np.array([KD.get(a, 0.0) for a in seq])
    k    = np.convolve(vals, np.ones(w)/w, mode="valid")
    return float(k.max())


def count_tm_segments(seq, w=19, thr=1.6):
    """Number of non-overlapping windows above the TM hydrophobicity threshold."""
    if len(seq) < w:
        return 0
    vals = np.array([KD.get(a, 0.0) for a in seq])
    k    = np.convolve(vals, np.ones(w)/w, mode="valid")
    above, i, n = 0, 0, len(k)
    while i < n:
        if k[i] >= thr:
            above += 1
            i += w
        else:
            i += 1
    return above


def signal_peptide_score(seq):
    """Heuristic SignalP-style score on the N-terminal 30 residues: a positively
    charged n-region, a hydrophobic h-region, and a small-residue c-region.
    Returns 0-1. A proper SignalP 6.0 run is better; this keeps the block
    dependency-free."""
    n = seq[:30]
    if len(n) < 15:
        return 0.0
    nreg = sum(1 for a in n[:5] if a in POSITIVE) / 5.0
    hreg = max_hydrophobic_window(n[5:25], w=7) if len(n) >= 25 else 0.0
    creg = sum(1 for a in n[-6:] if a in "AGSTC") / 6.0
    return float(np.clip((nreg + max(hreg, 0)/4.5 + creg) / 3.0, 0, 1))


def cleavage_sites(seq):
    """Protease cleavage positions. Rules as used in in-silico digestion:
      trypsin      after K or R, not before P
      chymotrypsin after F, W, Y (and L, M weakly), not before P
      pepsin       after or before F, L, W, Y at low pH
    """
    tryp = [i for i in range(len(seq)-1)
            if seq[i] in "KR" and seq[i+1] != "P"]
    chym = [i for i in range(len(seq)-1)
            if seq[i] in "FWYLM" and seq[i+1] != "P"]
    peps = [i for i in range(len(seq)-1)
            if seq[i] in "FLWY" or seq[i+1] in "FLWY"]
    return tryp, chym, peps


def fragment_stats(sites, L):
    """Peptide-length distribution implied by a cleavage-site list."""
    if L == 0:
        return 0, 0.0, 0
    cuts = [0] + sorted(set(sites)) + [L]
    frs  = np.diff(cuts)
    frs  = frs[frs > 0]
    if frs.size == 0:
        return 0, 0.0, 0
    # 4-20 residues is the window in which bioactive food peptides sit
    bioactive = int(((frs >= 4) & (frs <= 20)).sum())
    return len(frs), float(frs.mean()), bioactive


def motif_counts(seq):
    """Sorting and modification motifs relevant to EV cargo loading."""
    m = {}
    # N-glycosylation sequon N-X-S/T, X != P
    m["n_glyc"]   = len(re.findall(r"N[^P][ST]", seq))
    # O-glycosylation proxy: S/T inside a Pro-rich neighbourhood
    m["o_glyc"]   = len(re.findall(r"P.{0,2}[ST]|[ST].{0,2}P", seq))
    # YXXΦ — endosomal sorting, adaptor AP-2
    m["yxxphi"]   = len(re.findall(r"Y..[AVILMFWC]", seq))
    # PPXY — ESCRT / NEDD4 late domain, used by EV biogenesis
    m["ppxy"]     = len(re.findall(r"PP.Y", seq))
    # NPXY — clathrin-mediated internalisation
    m["npxy"]     = len(re.findall(r"NP.Y", seq))
    # Dileucine sorting signal
    m["dileucine"] = len(re.findall(r"[DE]..LL", seq))
    # KFERQ-like — chaperone-mediated autophagy targeting
    m["kferq"]    = len(re.findall(r"[QN][KR][ILVF][DE][QN]|[QN][DE][ILVF][KR][QN]", seq))
    # Phosphorylation context: S/T/Y in a basic or acidic neighbourhood
    m["phospho"]  = len(re.findall(r"[RK]..[ST]|[ST]..[DE]", seq))
    # Palmitoylation: Cys in the terminal 20 residues
    m["palmitoyl"] = sum(1 for a in seq[:20] + seq[-20:] if a == "C")
    # N-myristoylation: N-terminal Gly at position 2
    m["myristoyl"] = int(len(seq) > 1 and seq[1] == "G")
    return m


def reduced_composition(seq, scheme):
    L = len(seq)
    out = {}
    for name, group in scheme.items():
        out[name] = sum(1 for a in seq if a in group) / L if L else 0.0
    return out


# ── MAIN EXTRACTOR ────────────────────────────────────────────────────────────
def extract(seq):
    f = {}
    seq = str(seq).strip().upper()
    L   = len(seq)
    c   = Counter(seq)

    # ═══ COMPOSITION ═══════════════════════════════════════════════════════
    f["comp_length"]     = L
    f["comp_length_log"] = math.log1p(L)          # right-skewed; log is data-independent
    for a in AA:
        f[f"comp_aac_{a}"] = c[a] / L if L else 0.0
    for scheme_name, scheme in RED_SCHEMES.items():
        for k, v in reduced_composition(seq, scheme).items():
            f[f"comp_red_{scheme_name}_{k}"] = v

    # ═══ PHYSICOCHEMICAL ═══════════════════════════════════════════════════
    pa = ProteinAnalysis(seq)
    try:
        f["phys_mw"]     = pa.molecular_weight()
    except Exception:
        f["phys_mw"]     = sum(MW_AA.get(a, 110.0) for a in seq) - MW_WATER*(L-1)
    f["phys_mw_log"]     = math.log1p(f["phys_mw"])
    try:
        f["phys_pI"]     = pa.isoelectric_point()
    except Exception:
        f["phys_pI"]     = 7.0
    try:
        f["phys_gravy"]  = pa.gravy()
    except Exception:
        f["phys_gravy"]  = float(np.mean([KD.get(a, 0) for a in seq])) if L else 0.0
    try:
        f["phys_aromaticity"] = pa.aromaticity()
    except Exception:
        f["phys_aromaticity"] = sum(c[a] for a in "FWY") / L if L else 0.0
    try:
        f["phys_instability"] = pa.instability_index()
    except Exception:
        f["phys_instability"] = 40.0

    # Aliphatic index (Ikai) — a thermostability indicator
    if L:
        xA, xV = c["A"]/L, c["V"]/L
        xI, xL = c["I"]/L, c["L"]/L
        f["phys_aliphatic_index"] = 100*(xA + 2.9*xV + 3.9*(xI + xL))
    else:
        f["phys_aliphatic_index"] = 0.0


    for pH in (1.5, 2.0, 5.0, 6.7, 7.4):
        f[f"phys_charge_pH{pH}"] = net_charge(seq, pH)
    f["phys_charge_density_pH7.4"] = f["phys_charge_pH7.4"] / L if L else 0.0
    f["phys_charge_swing_2_to_7.4"] = f["phys_charge_pH2.0"] - f["phys_charge_pH7.4"]


    for name, group in [("hydrophobic", HYDROPHOBIC), ("polar", POLAR),
                        ("charged", CHARGED), ("positive", POSITIVE),
                        ("negative", NEGATIVE), ("aromatic", AROMATIC),
                        ("tiny", TINY), ("small", SMALL), ("disorder", DISORDER)]:
        f[f"phys_frac_{name}"] = sum(c[a] for a in group) / L if L else 0.0

    # Complexity
    f["phys_entropy"]          = shannon_entropy(seq)
    f["phys_low_complexity"]   = low_complexity_frac(seq)
    f["phys_longest_run"]      = longest_run(seq) if L else 0
    f["phys_longest_run_log"]  = math.log1p(f["phys_longest_run"])

    # ═══ STRUCTURAL ════════════════════════════════════════════════════════
    if L:
        f["struct_helix_propensity"] = float(np.mean([P_HELIX.get(a, 1.0) for a in seq]))
        f["struct_sheet_propensity"] = float(np.mean([P_SHEET.get(a, 1.0) for a in seq]))
        f["struct_turn_propensity"]  = float(np.mean([P_TURN.get(a, 1.0) for a in seq]))
    else:
        f["struct_helix_propensity"] = f["struct_sheet_propensity"] = \
            f["struct_turn_propensity"] = 1.0
    f["struct_helix_sheet_ratio"] = (f["struct_helix_propensity"] /
                                     max(f["struct_sheet_propensity"], 1e-6))
    try:
        h, t, s = pa.secondary_structure_fraction()
        f["struct_frac_helix"], f["struct_frac_turn"], f["struct_frac_sheet"] = h, t, s
    except Exception:
        f["struct_frac_helix"] = f["struct_frac_turn"] = f["struct_frac_sheet"] = 0.0

    f["struct_signal_peptide_score"] = signal_peptide_score(seq)
    f["struct_tm_potential"]         = max_hydrophobic_window(seq, 19)
    f["struct_tm_segments"]          = count_tm_segments(seq)
    f["struct_nterm_hydrophobic"]    = (float(np.mean([KD.get(a, 0) for a in seq[:25]]))
                                        if L >= 25 else 0.0)

    # ═══ FUNCTIONAL ════════════════════════════════════════════════════════
    mot = motif_counts(seq)
    for k, v in mot.items():
        f[f"func_{k}_count"]   = v
        f[f"func_{k}_density"] = v / L if L else 0.0

    f["func_cys_count"]     = c["C"]
    f["func_cys_frac"]      = c["C"] / L if L else 0.0
    # Disulfide potential: pairs of cysteines available to bond
    f["func_disulfide_pot"] = c["C"] // 2


    tryp, chym, peps = cleavage_sites(seq)
    for name, sites in [("trypsin", tryp), ("chymotrypsin", chym), ("pepsin", peps)]:
        nfrag, meanlen, bioact = fragment_stats(sites, L)
        f[f"func_{name}_sites"]        = len(sites)
        f[f"func_{name}_density"]      = len(sites) / L if L else 0.0
        f[f"func_{name}_n_fragments"]  = nfrag
        f[f"func_{name}_mean_fraglen"] = meanlen
        f[f"func_{name}_bioactive"]    = bioact

    f["func_bioactive_score"] = (f["func_trypsin_bioactive"] +
                                 f["func_chymotrypsin_bioactive"] +
                                 f["func_pepsin_bioactive"]) / L if L else 0.0


    cleav_density = (f["func_trypsin_density"] + f["func_chymotrypsin_density"]
                     + f["func_pepsin_density"]) / 3.0
    f["func_digestive_survival"] = float(np.clip(
        0.35 * (1 - min(cleav_density / 0.30, 1.0)) +
        0.20 * min(f["func_disulfide_pot"] / 10.0, 1.0) +
        0.20 * (1 - min(f["phys_instability"] / 60.0, 1.0)) +
        0.15 * min(f["struct_frac_helix"] + f["struct_frac_sheet"], 1.0) +
        0.10 * (1 - f["phys_low_complexity"]), 0, 1))


    f["func_stability_score"]  = float(np.clip(
        0.6 * (1 - min(f["phys_instability"] / 80.0, 1.0)) +
        0.4 * min(f["phys_aliphatic_index"] / 120.0, 1.0), 0, 1))
    f["func_predicted_stable"] = int(f["phys_instability"] < 40)

    return f


def dipeptide_composition(seq):
    L = len(seq)
    counts = Counter(seq[i:i+2] for i in range(L-1)
                     if seq[i] in AA and seq[i+1] in AA)
    tot = max(L - 1, 1)
    return {f"comp_dpc_{a}{b}": counts.get(a+b, 0) / tot for a in AA for b in AA}


# ── RUN ───────────────────────────────────────────────────────────────────────
df = pd.read_csv(INPUT_PATH)
missing = [c for c in META_COLS + [SEQ_COL] if c not in df.columns]
if missing:
    raise ValueError(f"Missing column(s): {missing}. Found: {list(df.columns)}")
if df[ID_COL].duplicated().any():
    raise ValueError(f"{ID_COL} is not unique")

print(f"Loaded {len(df):,} sequences from {INPUT_PATH}")
print(f"Length range: {df[SEQ_COL].str.len().min()}-{df[SEQ_COL].str.len().max()}\n")

rows = []
for i, s in enumerate(df[SEQ_COL].astype(str), 1):
    r = extract(s)
    if INCLUDE_DPC:
        r.update(dipeptide_composition(s.upper()))
    rows.append(r)
    if i % 1000 == 0:
        print(f"  {i:,} / {len(df):,}")

feat = pd.DataFrame(rows)
out  = pd.concat([df[META_COLS].reset_index(drop=True), feat], axis=1)


const = [c for c in feat.columns if feat[c].nunique(dropna=False) <= 1]
if const:
    out = out.drop(columns=const)
    print(f"\nDropped {len(const)} constant column(s)"
          + (f": {const[:6]}{'...' if len(const) > 6 else ''}" if len(const) <= 12 else ""))

out = out.replace([np.inf, -np.inf], np.nan)
if out.isna().any().any():
    bad = out.columns[out.isna().any()].tolist()
    print(f"NaNs in {len(bad)} column(s) -> filled with 0: {bad[:6]}")
    out = out.fillna(0.0)

N    = len(out)
path = os.path.join(OUTPUT_DIR, f"features_bio_{N}.csv")
out.to_csv(path, index=False)


blocks = {"comp_aac": "amino-acid composition", "comp_dpc": "dipeptide composition",
          "comp_red": "reduced alphabet", "comp_": "length",
          "phys_": "physicochemical", "struct_": "structural", "func_": "functional"}
counted, seen = {}, set()
for pref, label in blocks.items():
    cols = [c for c in out.columns
            if c.startswith(pref) and c not in seen and c not in META_COLS]
    seen.update(cols)
    if cols:
        counted[label] = len(cols)

print(f"\n{'='*66}")
print(f"  Saved -> {path}")
print(f"  {N:,} rows x {out.shape[1]} columns "
      f"({out.shape[1]-len(META_COLS)} features + {len(META_COLS)} id/label)")
print(f"{'='*66}")
for k, v in counted.items():
    print(f"    {k:<26} {v:>5}")

print(f"\n  NEXT — scaling goes INSIDE the fold, never here:")
print("    from sklearn.preprocessing import QuantileTransformer")
print("    qt = QuantileTransformer(output_distribution='normal',")
print("                             n_quantiles=min(1000, len(X_train)))")
print("    X_tr = qt.fit_transform(X_tr);  X_va = qt.transform(X_va)")
print("\n  To combine with ProtT5, merge on seq_id inside your pipeline:")
print("    X = prott5.merge(bio, on='seq_id')   # then transform in-fold")
print("\n  Expect comp_length, struct_signal_peptide_score, struct_tm_potential")
print("  and phys_frac_hydrophobic to be uninformative — the negatives were")
print("  matched on exactly those. That is the matching working, not a failure.")
